In [1]:
import os

if "notebooks" in os.getcwd():
	os.chdir("..")

import glob
import numpy as np 
import re 

from scipy.ndimage import rotate
from scipy import ndimage as ndi


import trimesh
#trimesh.interfaces.blender._blender_executable = r"C:\Program Files\Blender Foundation\Blender 4.1\blender.exe"
import pymeshlab as ml
import trimesh
import numpy as np

from skimage import io, filters, morphology
import matplotlib.pyplot as plt

%matplotlib qt

from geo2stl.geo2stl import *
from geo2stl import geo2stl as g2s
from geo2stl import sat2stl as s2s
from city2stl import dem2stl as d2s

import notebooks.figure as figs 

from numpy2stl.simplify import simplify_mesh_surfaces
from numpy2stl import writeOBJ, write3MF
import numpy2stl.puzzle as puzzle; import numpy2stl.view as view; import numpy2stl.verify as verify; import numpy2stl.simplify as simplify; import numpy2stl.boolean as boolean
from numpy2stl import numpy2stl, triangles_to_facets, writeSTL, polygon
from numpy2stl import Solid; import numpy2stl.solid as solid

from notebooks.experimental import  resize_max, rescale

In [2]:
import cv2
from city2stl import create

def create_dem_model(im, cut=True):

	width = im.shape
	model = create.get_landspace_model(im, None, 1, simplify=False)

	if cut:
		puzzle_model = puzzle.make_puzzle_model(width, b=200, m=50, base_n=10)
		models = boolean.cut_puzzle_pieces_manifold(model, puzzle_model)
		base_models = puzzle.make_base_border(width, b=200, m=50, base_n=10, height=1, offset_dist=5)
		models = models | base_models
	
	else:
		models = {"DEM":model}

	return models


def make_dem_image(target_bbox, dim=600, 
			depth_scale=1.0, water_scale = 0.1, sat_scale=200, 
			height=30, base=5, 
			subtract_water=True):

	(N, S, E, W) = target_bbox 
	im = stitch_tiles_no_rasterio(target_bbox)*1.0
	im[im<0] = im[im<0] * depth_scale

	w,h = im.shape
	
	if subtract_water:
		#Mass = 200
		#aegean = 1000

		sat = s2s.fetch_bbox_image( N,S,E,W, scale=sat_scale, dataset="esa")
		sat = sat.clip(0,100)
		water = 1.0*((sat==80) | (sat==0))
		water = filters.median(water, np.ones((3,3)))
		water = cv2.resize(water, (h,w), interpolation=cv2.INTER_LINEAR)
		water = water*im.ptp()*water_scale
		im = im - water

		plt.figure()
		plt.imshow(water)
	else:
		im[im>0] = im[im>0] + im.ptp()*water_scale

	im = proj_map_geo_to_2D(im,np.array((N, S, E, W)))
	im = im[:,~np.any(np.isnan(im),axis=0)]

	im = rescale(im, max_size=dim, height=height, base=base, clip=[.01,99.99], smooth=3)
	im = im.round(1)
	im = im[::-1]

	return im


In [3]:
%load_ext autoreload
%autoreload 2

#### Get data from source and place in root
https://www.gebco.net/data-products/gridded-bathymetry-data/gebco-2020

""

In [4]:
out_dir = "C:/Users/eac84/OneDrive/Documents/Projects/3D Maps/Map_files/"


# North America

## Applelachans 

In [ ]:
(N, S, E, W) =  50, 32,  -64, -88
target_bbox = np.array((N, S, E, W))

In [ ]:

result = stitch_tiles_no_rasterio(  target_bbox)
im = result.copy()*1.
im[im<0] = im[im<0]*0.1
im[im<0] = im[im<0] - np.ptp(im.ravel())*0.1

#im = proj_map_geo_to_2D(im,target_bbox)

im = rescale(im, max_size=900, height=25, base=0.1)
im = im[::-1]

figs.plot_data(im)
plt.imshow(result)

In [ ]:

fig , axs = plt.subplots(2,1, figsize=(12, 6), 
				layout = 'tight', 
				sharex=True, sharey=True)

im2 = im.copy()
remap = np.array([[0,0],[3,5],[7.5,6],[8,8.5],[9,9],[11,15],[15,21],[17,23],[30,30]])
im2[:,:,] = np.interp(im[:,:,],remap[:,0], remap[:,1])



axs = axs.ravel()
axs[0].hist(im.ravel()[::10],100)
axs[1].hist(im2.ravel()[::10],100)

fig , axs = plt.subplots(1,1, figsize=(12, 6), 
				layout = 'tight', 
				sharex=True, sharey=True)
axs.plot(remap[:,0],remap[:,1],"-o")
axs.set_aspect("equal")
print("")


In [ ]:
figs.plot_data(im)

In [ ]:
name = "Appalachia"
savefile(out_dir, name, im)

## Wiss

In [ ]:
(N, S, E, W) =  40.085,40.010, -75.182,   -75.235

## NORTH JERSEY

In [ ]:
(N, S, E, W) =  41.5, 40,  -73.5, -75.5
(N, S, E, W) =  41.5, 40.25,  -73.75, -75.25

In [ ]:
dim = 600

target_bbox = np.array((N, S, E, W))
N,S,E,W = (N*1. , S*1., E*1., W*1.)
bounds_NW = (N,S), (W,E) 

result = stitch_tiles_no_rasterio(  target_bbox)
im = proj_map_geo_to_2D(result,np.array((N, S, E, W)))

im = result.copy()
im[im<0] = im[im<0] - np.ptp(im.ravel())*0.01

if 1:

	sat = s2s.fetch_bbox_image( N,S,E,W, scale=100, dataset="esa")
	water = 20.0*(sat == 80)

	im = resize_max(im, max_size=dim)
	water = resize_max(water, max_size=dim)

	water = water.clip(0,10)*2
	im = im - water
	
im = rescale(im, max_size=dim, height=25, base=0.2, clip=None)

x = np.array(target_bbox)[np.array([3,2,1,0])]
figs.plot_data(im, bbox = x )
plt.figure()
plt.imshow(water)


In [ ]:
#im2 = scipy.ndimage.median_filter(im, 3)
models = {}
vertices, faces = get_landspace_model(im, bounds_NW, 500, simplify=False)
mesh = trimesh.Trimesh(vertices=vertices, faces=faces)
mesh.fix_normals()
models["landscape"] = mesh.vertices, mesh.faces


In [ ]:
view.render_models_napari( models)

In [ ]:

x = 15*((im/10)+1).round(0)
vx,fs = get_landspace_model(x, bounds_NW, 500, simplify=False)
mesh = trimesh.Trimesh(vx, fs)
mesh.fix_normals()
vx,fs  = mesh.vertices, mesh.faces

model_1 = {"Before":(vx, fs)}

verify.check_model_status( model_1 )


In [ ]:
plt.close("all")
mesh = trimesh.Trimesh(vx, fs)
mesh.fix_normals()
vx,fs  = mesh.vertices, mesh.faces
fs2 = simplify_mesh_surfaces(*model_1["Before"])
##########################
mesh = trimesh.Trimesh(vertices=vx, faces=fs2)
mesh.fix_normals()
vx2,fs2  = mesh.vertices, mesh.faces
model = {"After":(vx2, fs2)}

verify.check_model_status( model )

print(len(fs), len(fs2))
view.render_models_napari( model )


In [ ]:
view.render_models_napari( model | model_1 )

In [ ]:
%lprun -f solid.vertices_to_index get_landspace_model(x, bounds_NW, 500, simplify=False)

In [ ]:
_, fs2 = simplify_mesh_surfaces(*models_land["landscape"])

verify.check_model_status({"test":(vx,fs2)} )
print(len(models_land["landscape"][1]))
print(len(fs2))

In [ ]:
width = im.shape
b = 200
m = 50        # Distance from corners to start the notch
base_n = 10  
puzzle_models = puzzle.make_puzzle_model(width, b=500, m=50, base_n=10)

model = models["landscape"]
puzzle_cut = puzzle.cut_puzzle_pieces(model, puzzle_models)

view.render_models_napari(puzzle_cut)

verify.check_model_status(puzzle_cut)
writeOBJ("test.obj", puzzle_cut)


## Penn

In [7]:
(N, S, E, W) =  42.8, 39.5,  -73.75, -80.


In [8]:

target_bbox = (N, S, E, W)

result = stitch_tiles_no_rasterio( target_bbox)
im = proj_map_geo_to_2D(im,np.array((N, S, E, W)))

im = result.copy()

im[im<0] = im[im<0]- np.ptp(im.ravel())*0.05

im = rescale(im, max_size=800, height=35, base=0.1,
         clip=None , smooth=[.1,99.95])

==== Stitching tiles ====
Target bounding box: (42.8, 39.5, -73.75, -80.0)
✅ Finished stitching


NameError: name 'im' is not defined

In [ ]:
name = "Penn"
write.savefile(out_dir, name, im)

## Vermont 

In [10]:
(N, S, E, W) =  45.3, 42.5, -72.05 , -73.65
#(N, S, E, W) =  45.3, 42.6, -72.01 , -73.54 
(N, S, E, W) =  45.4, 42.45, -71.72 , -73.8 

name = "Vermont"
dim = 800
depth_scale = 0.5
bbox = [W,E,S,N]
target_bbox = (N, S, E, W)

plt.close("all")
im = make_dem_image(target_bbox, dim=dim, 
					depth_scale=0.5, water_scale = 0.05, 
                    base = 10, height=20,
					subtract_water=True)
figs.plot_data(im, bbox=bbox,  close=False)


if 0:
	width = im.shape
	model = create.get_landspace_model(im, None, 1, simplify=False)

	puzzle_model = puzzle.make_puzzle_model(width, b=200, m=50, base_n=10)
	models = boolean.cut_puzzle_pieces_manifold(model, puzzle_model)
	base_models = puzzle.make_base_border(width, b=200, m=50, base_n=10, height=1, offset_dist=5)
	models = models | base_models

	out_dir2 = out_dir + "/" + name 
	os.makedirs(out_dir2,exist_ok=True)
	filename = out_dir2 + "/" + name + ".3mf"
	write3MF(filename, models)

==== Stitching tiles ====
Target bounding box: (45.4, 42.45, -71.72, -73.8)
✅ Finished stitching
Loading esa from cache...
(800, 396)


In [174]:
%load_ext line_profiler

In [ ]:
figs.plot_data(im, bbox=bbox,  close=True)

In [ ]:
(N, S, E, W) =  45.3, 42.5, -70.66 , -73.64 ## White Mountains 

name = "Green_White_mts"
dim = 800
depth_scale = 0.5


target_bbox = (N, S, E, W)
im = make_dem_image(target_bbox, dim=dim, depth_scale=0.5, subtract_water=True)
figs.plot_data(im)

## Mass

In [ ]:
(N, S, E, W) =  43.5, 40.9, -69.5, -73.5
name = "Mass"
dim = 600
depth_scale = 0.5
target_bbox = (N, S, E, W)

In [ ]:
im = make_dem_image(target_bbox, dim=dim, depth_scale=0.5, subtract_water=True)
#figs.plot_data(im)

In [ ]:

models = create_dem_model(im,target_bbox)

out_dir2 = out_dir + "/" + name 
os.makedirs(out_dir2,exist_ok=True)
filename = out_dir2 + "/" + name + "_1.3mf"
write3MF(filename, models)

In [ ]:
verify.check_model_status(models)
view.render_models_napari(models)

In [ ]:
base_models = puzzle.make_base_border((100,100), b=200, m=50, base_n=10, height=1, offset_dist=5)

In [ ]:
pts_list = puzzle.make_puzzle_pts((400,400), b=200, m=50, base_n=10, tol=0.3)
plt.figure()
for pts in pts_list:
  plt.plot(pts[:,0],pts[:,1])
  plt.axis("equal")

## Great Lakes

In [ ]:
plt.imshow(im)

In [ ]:

target_bbox = (N, S, E, W)

result = stitch_tiles_no_rasterio(  target_bbox)


im = result.copy()
im[im<0] = im[im<0]*0.1

im = proj_map_geo_to_2D(im,np.array((N, S, E, W)))

im = im[:,~np.any(np.isnan(im),axis=0)]
im = im * 0.1

im = rescale(im, max_size=800, height=25, base=0.1,
         clip=[.01,99.99], smooth=3)



In [ ]:
figs.plot_data(im)

In [ ]:

# Simplify mesh to 10% of original face count
mesh = trimesh.Trimesh(vertices=vertices, faces=faces)
simplified = mesh.simplify_quadric_decimation(0.9)
vertices = simplified.vertices.copy().astype(float)
faces = simplified.faces
#
simplified.export(out_dir + "simplified.stl")

## INDIAN RIVER

In [ ]:
N,S,E,W = 28,27,-80,-80.6

In [ ]:
target_bbox = (N, S, E, W)
result = stitch_tiles_no_rasterio( target_bbox)

im[im>0] = im[im>0] + 10
im[im<-20] = -20



## Colorado

In [ ]:
N,S,E,W = [  41.0034002,   36.9925245, 
       	   -102.041585 , -109.0601879]


In [ ]:
target_bbox = np.array((N, S, E, W))
NSEW = np.array([N,S,E,W])

result = stitch_tiles_no_rasterio(  target_bbox)
im = result.copy()

x = 1500
im[im>x] = 0.3*(im[im>x]-x)+x

im = im * 0.02

im = resize_geo_aspect(im, NSEW)

filt = ndi.gaussian_filter(im, sigma=50)
im = im - filt*0.7

im = rescale(im, max_size=800, height=50, base=0.1,
         clip=[.01,99.99], smooth=3)



In [ ]:
name = "Colorado1"
savefile(out_dir, name, im)

## Grand Canyon

In [ ]:
N,S,E,W = [ 36.9, 35.5, -111.5, -114.2]
N,S,E,W = [ 36.9, 35.5, -111.5, -115.0] # With LakeMead
name = "GrandCanyon"
dim = 800
depth_scale = 0.5


In [ ]:
target_bbox = (N, S, E, W)
bbox = [W,E,S,N]

im = make_dem_image(target_bbox, dim=dim, depth_scale=0.5, subtract_water=True)
figs.plot_data(im, bbox=bbox)

In [ ]:
models = create_dem_model(im)
out_dir2 = out_dir + "/" + name 
os.makedirs(out_dir2,exist_ok=True)
filename = out_dir2 + "/" + name + ".3mf"
write3MF(filename, models)

## Pudget Sounds

In [1]:
N,S,E,W = [ 48.5, 46.5, -121.60 , -123.5 ] 

N,S,E,W = [ 48.5, 46.8, -121.0 , -123.5 ] # BIG

name = "PugetSound"
dim = 800
depth_scale = 0.5
target_bbox = (N, S, E, W)
bbox = [W,E,S,N]

im = make_dem_image(target_bbox, dim=dim, 
                    depth_scale=1, 
                    water_scale = 0.05,
					subtract_water=True)
figs.plot_data(im, bbox=None, close=True)

NameError: name 'make_dem_image' is not defined

# Central America

## Caribbean

In [ ]:
N,S,E,W = 31,6,-54,-102

In [ ]:
target_bbox = np.array((N, S, E, W))
result = stitch_tiles_no_rasterio( target_bbox)

im = result.copy()
im[im>0] = im[im>0]+1000

#im3 = proj_map_height(im2, np.array([n,s,e,w]))
im = proj_map_geo_to_2D(im, target_bbox )

im = rescale(im, max_size=1000, height=30, base=0.1,
         clip=[.01,99.99], smooth=None)


In [ ]:
tri = np3D.numpy2stl(im4)
vertices, faces = np3D.vertices_to_index(tri)
mesh = trimesh.Trimesh(vertices,faces)
mesh.show()

In [ ]:
facets = np3D.triangles_to_facets(tri)
np3D.writeSTL(facets, "Ocean_flat.stl")

## Puerto Rico

In [ ]:
N,S,E,W  = 18.75, 17.65, -65.15, -67.35

In [ ]:
target_bbox = np.array((N, S, E, W))
result = stitch_tiles_no_rasterio(  target_bbox)
im = result.copy()

im[im<0] = im[im<0]*0.2
im[im>0] = im[im>0] + 100

im_x = im - im.min() + (im.ptp()*0.1)

scale = 2
sz = tuple((np.array(im_x.shape)[[1,0]]*scale).astype(int))
im_x = im_x * scale
im_x = cv2.resize(im_x,sz)
im_x = im_x * 0.02


In [ ]:
name = "PuertoRico"
savefile(out_dir, name, im_x)

## Keys

In [ ]:
(N, S, E, W) = 25.9,24.3, -80,-82.0

In [ ]:
target_bbox = (N, S, E, W)

result = stitch_tiles_no_rasterio( target_bbox)

im = result.copy()
im[im<0] = im[im<0]*0.1

im = proj_map_geo_to_2D(im,np.array((N, S, E, W)))

im = im[:,~np.any(np.isnan(im),axis=0)]

im[im<-100] = -100
im[im<0] = im[im<0]*0.1
im[im>0] = im[im>0] + 10

#im[im<0] = im[im<0]-200

scale = 2
im_x = im.copy()
sz = tuple((np.array(im_x.shape)[[1,0]]*scale).astype(int))
im_x = im_x * scale
im_x = cv2.resize(im_x,sz)
im_x = im_x * 0.02

DEM = im_x
rotation = 55 - 90

DEM = rotate(DEM,rotation, reshape=True, cval=-10)

DEM = DEM[480:820,:]
DEM[DEM==-10] = DEM[DEM>-10].min()




## Bahamas

In [ ]:
N,S,E,W = 35,10,-60,-88


In [ ]:

target_bbox = np.array((N, S, E, W))
result = stitch_tiles_no_rasterio( target_bbox)

im = result.copy()

im = proj_map_geo_to_2D(im, target_bbox , clip_out=False)
im[np.isnan(im)] = -10000

DEM = im.copy()
lo,hi = np.percentile(DEM.ravel(), [.01,99.99])
DEM = DEM.clip(lo,hi)

#DEM = DEM[::10,::10]
rotation = 38

DEM = rotate(DEM,rotation, reshape=True, cval=-0)

x1,x2,y1,y2 = 3800,6100, 2200,6700

DEM2 = DEM[x1:x2, y1:y2].copy()

lo,hi = np.percentile(DEM.ravel(), [.01,99.99])
DEM = DEM.clip(lo,hi)

DEM2[DEM2>0] = DEM2[DEM2>0]+np.ptp(DEM2)*0.1
DEM2 = resize_max(DEM2, max_size=1000)
DEM2 = DEM2 - DEM2.min() + DEM2.ptp()*0.1
DEM2 = DEM2 / DEM2.ptp() * 25

plt.close("all")	
plt.figure()
plt.imshow(DEM,cmap="jet")
plt.plot([y1,y1,y2,y2,y1],[x1,x2,x2,x1,x1], color="white", linewidth=3)
plt.grid(True)

plt.figure()
plt.imshow(DEM2,cmap="jet")
plt.grid(True)


plt.figure()
_ = plt.hist(DEM2.ravel(), bins=100)

In [ ]:
name = "Antillas_38deg"
savefile(out_dir, name, DEM2)

## Mexico

In [ ]:
name = "Mexico"
N,S,E,W = 34.0,13.0,-84.0,-118.5
dim = 600

target_bbox = np.array((N, S, E, W))

im = make_dem_image(target_bbox, dim=dim, depth_scale=0.25, water_scale=0.05, sat_scale=1000, subtract_water=True)

In [ ]:
figs.plot_data(im, bbox = [W,E,S,N] , close=True)

In [ ]:
models = create_dem_model(im)
out_dir2 = out_dir + "/" + name 
os.makedirs(out_dir2,exist_ok=True)
filename = out_dir2 + "/" + name + ".3mf"
write3MF(filename, models)

## Panama

In [1]:
name = "Panama"

N,S,E,W = 11.4, 7, -74.5, -86.  # CostaRica + Colombia
N,S,E,W = 11.4, 6.7, -76.5, -86.1  # CostaRica
#N,S,E,W = 10, 7, -77, -81.6  # Panama_only

dim = 800

target_bbox = np.array((N, S, E, W))

im = make_dem_image(target_bbox, dim=dim, 
                    depth_scale=0.5, water_scale=0.05, sat_scale=600, 
					height=30, base=10, 
					subtract_water=True)
figs.plot_data(im, bbox = [W,E,S,N] , close=True)
if 0:
	models = create_dem_model(im)
	out_dir2 = out_dir + "/" + name 
	os.makedirs(out_dir2,exist_ok=True)
	filename = out_dir2 + "/" + name + ".3mf"
	write3MF(filename, models)

NameError: name 'np' is not defined

## Hawaii

In [10]:
name = "Hawaii"


N,S,E,W = 23.7, 17.5, -154., -161.5

dim = 600

target_bbox = np.array((N, S, E, W))

im = make_dem_image(target_bbox, dim=dim, 
                    depth_scale=1, water_scale=0.05, sat_scale=600, 
					height=30, base=10, 
					subtract_water=True)
figs.plot_data(im, bbox = [W,E,S,N] , close=True)
if 0:
	models = create_dem_model(im)
	out_dir2 = out_dir + "/" + name 
	os.makedirs(out_dir2,exist_ok=True)
	filename = out_dir2 + "/" + name + ".3mf"
	write3MF(filename, models)

==== Stitching tiles ====
Target bounding box: [  23.7   17.5 -154.  -161.5]
Finished stitching
(541, 600)


# South America

## Colombia

In [ ]:
from city2stl import osm2stl as o2s
from city2stl import create 

#from sd2s import embed_lines
from skimage import morphology
from skimage.draw import line_aa, polygon2mask

In [ ]:
N,S,E,W = 14,-6,-64,-84
target_bbox = (N, S, E, W)
result = stitch_tiles_no_rasterio( target_bbox)

In [ ]:

nsew = np.array([N,S,E,W])

x1 = 300 
x3 = 2000
im_ = im.copy()
im_[im_>x1] = im_[im_>x1] + (x3-x1)
bx = (im_<=x1) & (im_>0)
im_[bx] = im_[bx] * (x3/x1)

im_[im_<0] = im_[im_<0]*0.5
im_[im_>0] = im_[im_>0] + 10

#############
scale = 0.5
sz = tuple((np.array(im_.shape)[[1,0]]*scale).astype(int))
im_ = im_ * scale
im_ = cv2.resize(im_,sz)
im_ = im_ * 0.01

###############
sealine = im_>0
sealine = morphology.binary_dilation(sealine, morphology.disk(2)) & ~sealine
im_ = im_ - (sealine* im_.ptp()*0.01)

################
im_lines = get_border(nsew, "Colombia", im_.shape)
im_lines[im_<0] = 0 
im_ = im_ - (im_lines* im_.ptp()*0.01)

################
im_ = im_ - np.min(im_) + (im_.ptp()*.01)

In [ ]:

name = "Colombia"
savefile(out_dir, name, im_ )

## Medellin

In [51]:
N,S,E,W = [6.5, 5.5, -74.9, -75.9]

name = "Medellin"
dim = 500
target_bbox = (N, S, E, W)
bbox = [W,E,S,N]

im = make_dem_image(target_bbox, dim=dim, 
                    depth_scale=0.5, 
                    water_scale = 0.05,
                    height=20, 
					subtract_water=False)
figs.plot_data(im, bbox=bbox, close=True)

if 0:
	models = create_dem_model(im)
	out_dir2 = out_dir + "/" + name 
	os.makedirs(out_dir2,exist_ok=True)
	filename = out_dir2 + "/" + name + ".3mf"
	write3MF(filename, models)

==== Stitching tiles ====
Target bounding box: (6.5, 5.5, -74.9, -75.9)
✅ Finished stitching
(500, 495)


## GALAPAGOS

In [ ]:
N,S,E,W = [ 1.1,-2.1, -88.3, -92.1] 
N,S,E,W = [ 1.0,-2.0, -89.0, -92.0] 

name = "Galapagos"
dim = 500
target_bbox = (N, S, E, W)
bbox = [W,E,S,N]

im = make_dem_image(target_bbox, dim=dim, 
                    depth_scale=0.5, 
                    water_scale = 0.05,
                    height=20, 
					subtract_water=True)
figs.plot_data(im, bbox=bbox, close=True)

if 0:
	models = create_dem_model(im)
	out_dir2 = out_dir + "/" + name 
	os.makedirs(out_dir2,exist_ok=True)
	filename = out_dir2 + "/" + name + ".3mf"
	write3MF(filename, models)

## Brazil

In [ ]:
im_all = io.imread(tile_files[1])
plt.imshow(im_all[::10,::10])

In [ ]:
index_list = [
[5500,5800,10200,10700],
[5420,5550,11150,11350],
[5200,5800,10200,11500],
[4000,7000,7500,10500],
[5800,6500,8000,9000],
[5900,6200,8300,8800,]
]

plt.close("all")
plt.figure()
plt.imshow(im_all, cmap="jet")
for index in index_list:
	plt.plot(
		[index[2],index[2],index[3],index[3],index[2]],
		[index[0],index[1],index[1],index[0],index[0]], color="white", linewidth=3)
#plt.grid(True)


for index in index_list:
	imx = im_all[index[0]:index[1],index[2]:index[3]]*1.
	plt.figure()
	plt.imshow(imx, cmap="jet")

## Amazon

In [101]:
N,S,E,W = 15,-19,-45,-85

#N,S,E,W = 14,-20,-45,-80
name = "Amazon"
target_bbox = (N, S, E, W)
dim = 600
bbox = [W,E,S,N]

In [114]:
target_bbox = (N, S, E, W)
result = g2s.stitch_tiles_no_rasterio( target_bbox)

im = result.copy()*1.
im2 = (im.copy())

#im2[im2>x] = ((im2[im2>x] - x)*0.1)+x
im2[im2<0] = im2[im2<0]*0.2
im2[im2>0] = (1*im2[im2>0] / np.max(im))**.4 * np.max(im) 

im = im2.copy()
im = proj_map_geo_to_2D(im,np.array((N, S, E, W)))
im = im[:,~np.any(np.isnan(im),axis=0)]

height = 30
base = 10
im = rescale(im, max_size=dim, height=height, base=base, clip=[.01,99.99], smooth=3)

figs.plot_data(im[::-1], bbox=bbox, close=False)

==== Stitching tiles ====
Target bounding box: (15, -19, -45, -85)
✅ Finished stitching
(539, 600)


In [116]:
models = create_dem_model(im[::-1], cut=True)
out_dir2 = out_dir + "/" + name 
os.makedirs(out_dir2,exist_ok=True)
filename = out_dir2 + "/" + name + ".3mf"
write3MF(filename, models)

Creating top...Creating walls...Creating bottom...
0.4
0
1
2
3
4
5
6
7
8
0.4
✅ Successfully saved 18 objects to C:/Users/eac84/OneDrive/Documents/Projects/3D Maps/Map_files//Amazon/Amazon.3mf


In [6]:
N,S,E,W = 10,-0,-65,-75

img = s2s.get_aquatic_regions(N,S,E,W, dataset="jrc", scale=500)
plt.imshow(img)

Loading jrc from cache...
...
(2227, 2227)


In [105]:
plt.figure()
x = np.linspace(0,1,100)

As = np.linspace(0.01,10,5)
plt.figure()
for a in As:
	plt.plot(x, a*x**0.5)

As = np.linspace(0.01,10,5)
plt.figure()
for a in As:
	plt.plot(x, x**a)

In [ ]:



#savefile(out_dir, name, im)

==== Stitching tiles ====
Target bounding box: (15, -20, -45, -85)
✅ Finished stitching
Loading jrc from cache...
...
(1949, 2228)


## Chile

In [135]:
N,S,E,W = -14,-58,-52,-85

name = "Chile"
target_bbox = (N, S, E, W)
dim = 600
bbox = [W,E,S,N]

In [136]:
im = make_dem_image(target_bbox, dim=dim, 
                    depth_scale=0.5, 
                    water_scale = 0.05,
                    height=20, 
					subtract_water=False)
figs.plot_data(im, bbox=bbox, close=True)

==== Stitching tiles ====
Target bounding box: (-14, -58, -52, -85)
✅ Finished stitching
(600, 238)


# Europe

## Norway

In [ ]:
N=72.07
S=57.03
E= 33.28 
W= -3

In [ ]:
target_bbox = (N, S, E, W)

result = g2s.stitch_tiles_no_rasterio( target_bbox)

im = result.copy()
im[im<0] = im[im<0]*0.1

im = proj_map_geo_to_2D(im,np.array((N, S, E, W)))

im = resize_max(im, max_size=800)

## North Sea

In [ ]:
N, S, E, W =67, 53, 38, -2

In [ ]:
target_bbox = (N, S, E, W)
result = g2s.stitch_tiles_no_rasterio( target_bbox)
im = result.copy()

if 1:
	im[im<0] = np.sqrt(np.abs(im[im<0])) * np.sign(im[im<0])
	im[im<0] = im[im<0] / (np.ptp(im[im<0])) * result[im<0].ptp()*0.5
	
im[im>0] = im[im>0] + np.ptp(im.ravel())*0.05

im = proj_map_geo_to_2D(im,np.array((N, S, E, W)), clip_out=True)
im = rescale(im, max_size=800, height=25, base=0.1, clip= [.01,99.99], smooth=3)

In [ ]:
name = "NorthSea"
savefile(out_dir, name, im)

## IBERIA

In [ ]:
N,S,E,W = 44.4, 34.8, 5.3,-11

In [ ]:
target_bbox = np.array((N, S, E, W))
nsew = np.array((N, S, E, W))
result = stitch_tiles_no_rasterio( target_bbox)
im = result.copy()

im[im<0] = im[im<0]*0.25
im[im>0] = im[im>0]+100

im_r = im.copy()
im_r[im_r<0]=0
im_r = (ndi.gaussian_filter(im_r,10) - im_r).clip(0)
im = im - 2*im_r


im2 = proj_map_geo_to_2D(im, np.array(nsew))
im2 = im2 - np.min(im2) + 1


scale = 0.3
sz = tuple((np.array(im2.shape)[[1,0]]*scale).astype(int))
im2 = im2 * scale
im2 = cv2.resize(im2,sz)
im2 = im2 * 0.02


In [ ]:
figs.plot_data(im2)

In [ ]:
name = "Ibera"
savefile(out_dir, name, im2 )


## Mediterrian 

In [ ]:
N, S, W, E =  48.5, 28.5, -15, 45.2

In [ ]:
target_bbox = (N, S, E, W)
result = stitch_tiles_no_rasterio( target_bbox)

im = result.copy()
im = proj_map_geo_to_2D(im,np.array((N, S, E, W)), True)
im = resize_max(im, max_size=800)

lo,hi = np.percentile(im.ravel(), [5,99.99])
im = im.clip(lo,hi)

im[im<0] = im[im<0]*0.8
im[im<0] = im[im<0]-im.ptp()*0.05

im = im / im.ptp()*30
im  = im - im.min() + im.ptp()*0.05


In [ ]:
figs.plot_data(im)

In [ ]:
view3D_napari(im)

In [ ]:
name = "Mediterranean"
savefile(out_dir, name, im)

## Aegean

In [ ]:
(N, S, E, W) =  43.0, 34.0, 30.9, 18.6

name = "Aegean"
dim = 600
depth_scale = 0.5
sat_scale = 400

target_bbox = (N, S, E, W)
im = make_dem_image(target_bbox, dim=dim, depth_scale=depth_scale, sat_scale=sat_scale,subtract_water=True)

models = create_dem_model(im)
out_dir2 = out_dir + "/" + name 
os.makedirs(out_dir2,exist_ok=True)
filename = out_dir2 + "/" + name + ".3mf"
write3MF(filename, models)

## Gibraltar

In [6]:
(N, S, E, W) =  37.83, 34.29, -2.05, -7.05
(N, S, E, W) =  37.94, 34.0, -1.7, -6.7


name = "Gibraltar"
dim = 600
depth_scale = 0.5
sat_scale = 400
water_scale = 0.05
target_bbox = (N, S, E, W)
bbox = np.array(target_bbox)[np.array([3,2,1,0])]
plt.close("all")
im = make_dem_image(target_bbox, dim=dim, 
						depth_scale=depth_scale, water_scale=water_scale, 
						sat_scale=sat_scale,
						base=10, height=30,
						subtract_water=True)

figs.plot_data(im, bbox=bbox, close=False)

if 0:
	models = create_dem_model(im)
	out_dir2 = out_dir + "/" + name 
	os.makedirs(out_dir2,exist_ok=True)
	filename = out_dir2 + "/" + name + ".3mf"
	write3MF(filename, models)

==== Stitching tiles ====
Target bounding box: (37.94, 34.0, -1.7, -6.7)
✅ Finished stitching
Cache paths -> data: c:\Users\eac84\OneDrive\Documents\Projects\3D Maps\Code\cache\ee\d8866754d9f6fa2bb4ade879caef2cf8.jbl, meta: c:\Users\eac84\OneDrive\Documents\Projects\3D Maps\Code\cache\ee\d8866754d9f6fa2bb4ade879caef2cf8.meta, exists: False
Caching esa at scale=400
(599, 600)


# AFRICA

## Pan Africa

In [ ]:
index = [11000,31500,15500,36000]


x = [0,21600*2]
y = [-90.0,90.0]

coor = np.interp(index, x,y)
coor[:2] = -coor[:2]
print(coor)

[ 44.16666667 -41.25       -25.41666667  60.        ]


: 

In [ ]:
im_4 = io.imread(tile_files[2])
im_1 = io.imread(tile_files[5])
im_2 = io.imread(tile_files[6])
im_3 = io.imread(tile_files[1])

im_all = np.vstack([np.hstack([im_1,im_2]),np.hstack([im_3,im_4])])
im_0 = im_all[index[0]:index[1],index[2]:index[3]] * 1.0



In [ ]:
(N, S, E, W) =  44.17, -41.25, 60, -25.41
target_bbox = coor[[0,1,3,2]]
result = g2s.stitch_tiles_no_rasterio( target_bbox)


In [ ]:
plt.figure()
plt.imshow(result[::10,::10]/(result.max()))
plt.figure()
plt.imshow(im_0[::10,::10]/(im_0.max()))

In [ ]:
im_0.shape, result.shape

In [ ]:
im_x = im_0
sz = tuple((np.array(im_x.shape)[[1,0]]*0.02).astype(int))
im_x = im_x * 0.1
im_x = cv2.resize(im_x,sz)*1.0

sz = tuple((np.array(im_x.shape)[[1,0]]*2).astype(int))
im_x = cv2.resize(im_x,sz)


im_x[im_x<0] = im_x[im_x<0]*.1
im_x[im_x<0] = im_x[im_x<0]*.2

x = 200
im_x[im_x>x] = 0.2*(im_x[im_x>x]-x)+x

x = 100
im_x[im_x>x] = 0.5*(im_x[im_x>x]-x)+x

x = 50
im_x[im_x>x] = 0.5*(im_x[im_x>x]-x)+x

x = -10
im_x[im_x<x] = 5*(im_x[im_x<x]-x)+x



im_x[im_x<0] = im_x[im_x<0] - 0.05*(im_x.max()-im_x.min())

n1,s1,e1,w1 = mat2coor( [0, 180, -90,90], [43200, 43200], index)
s1 = s1 - 180

nsew1 = (n1,s1,e1,w1)

im_x2 = proj_map_geo_to_2D(im_x,np.array(nsew1))

im_x = im_x - im_x.min() + 10

In [ ]:

name = "Africa"
savefile(out_dir, name, im_x)

## Cabo Verde

In [ ]:
(N, S, E, W) =  18.0, 14.0, -21.5, -26.

name = "CaboVerde"
dim = 600
depth_scale = 0.5
sat_scale = 400

target_bbox = (N, S, E, W)
im = make_dem_image(target_bbox, dim=dim, 
		depth_scale=depth_scale, sat_scale=sat_scale,
		height=20
		subtract_water=True)

figs.plot_data(im, bbox=None, close=True)

if 0:
	models = create_dem_model(im)
	out_dir2 = out_dir + "/" + name 
	os.makedirs(out_dir2,exist_ok=True)
	filename = out_dir2 + "/" + name + ".3mf"
	write3MF(filename, models)

==== Stitching tiles ====
Target bounding box: (18.0, 14.0, -21.5, -26.0)
✅ Finished stitching
caching
(559, 600)


# Asia 

## Bengal

In [ ]:
N,S,E,W =  29, 20 ,95, 84

In [ ]:
target_bbox = (N, S, E, W)
result = g2s.stitch_tiles_no_rasterio( target_bbox)
img = s2s.get_aquatic_regions(N,S,E,W, dataset="jrc")

im = result.copy()
img2 = img.copy().astype(np.uint8)

im = resize_max(im, max_size=1200)
img2 = filters.median(img2, np.ones((3,3)))
img2 = cv2.resize(img2, (im.shape[1], im.shape[0]), interpolation=cv2.INTER_LINEAR).astype(int)
img2[im<0] = 200
img2[im>500] = 0
img2 = img2 / 100
img2 = (img2).clip(0,1)


x = 1
xmax = im.max()
im[im>x] = ((im[im>x]/im.max())**0.7 )*im.max()


im = im  - img2*np.ptp(im.ravel())*0.1

im = g2s.proj_map_geo_to_2D(im,np.array((N, S, E, W)))

im = resize_max(im, max_size=600)
im = filters.median(im, np.ones((3,3)))
im = im.clip(*np.percentile(im.ravel(), [.01,99.99]))
im  = im - im.min() + im.ptp()*0.2
im = im / im.ptp() * 30




In [ ]:
name = "Bengal"
savefile(out_dir, name, im)

## Persia

In [ ]:
N,S,E,W = 41,  24 ,65, 41


In [ ]:

target_bbox = (N, S, E, W)
result = g2s.stitch_tiles_no_rasterio( target_bbox)

im = result.copy()
im = g2s.proj_map_geo_to_2D(im,np.array((N, S, E, W)))


im[im<0] = im[im<0]*0.2
im[im<0] = im[im<0]- np.ptp(im.ravel())*0.05
im  = im - im.min() + im.ptp()*0.1

im = resize_max(im, max_size=900)
im = filters.median(im, np.ones((3,3)))
lo,hi = np.percentile(im.ravel(), [.1,99.9])
im = im.clip(lo,hi)




In [ ]:
name = "Persia"
savefile(out_dir, name, im)

## Japans

In [ ]:
N,S,E,W = 55.066, 28.75, 153.69, 118
N,S,E,W = 48, 30, 150, 122

In [ ]:
target_bbox = (N, S, E, W)
result = g2s.stitch_tiles_no_rasterio( target_bbox)

im = result.copy()

im = g2s.proj_map_geo_to_2D(im,np.array((N, S, E, W)))

im = im[:,~np.any(np.isnan(im),axis=0)]

im[im<0] = im[im<0]*0.2
im[im<0] = im[im<0]- np.ptp(im.ravel())*0.05
im[im<0] = im[im<0].clip(-4000)

im = resize_max(im, max_size=800)
im = filters.median(im, np.ones((3,3)))

im  = im - im.min() + im.ptp()*0.1
im = im / im.ptp() * 30

In [ ]:
figs.plot_data(im)

In [ ]:
name = "Japans"
savefile(out_dir, name, im)

## South China Sea

In [ ]:
N,S,E,W = 25.938, -11.716, 133.769, 91.58
N,S,E,W = 25.938, -11.716, 136, 88
N,S,E,W = 29, -14, 140 , 86

In [ ]:
target_bbox = (N, S, E, W)
result = g2s.stitch_tiles_no_rasterio(target_bbox)

im = result.copy()
im = g2s.proj_map_geo_to_2D(im,np.array((N, S, E, W)))
im = resize_max(im, max_size=800)

lo,hi = np.percentile(im.ravel(), [.1,99.9])
im = im.clip(lo,hi)

im = filters.median(im, np.ones((3,3)))

im[im<0] = im[im<0]*0.8
im[im<0] = im[im<0] - im.ptp()*0.05

im  = im - im.min() + im.ptp()*0.1
im = im / im.ptp() * 30


In [ ]:

name = "SouthChinaSea"
savefile(out_dir, name, im)